# 03 Classification Features - Similarity And Topics

Purpose: inspect model-facing features created for classification, especially nearest-neighbor similarity and topic features.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 120)
plt.style.use('default')

DATA_DIR = Path('../data') if Path('../data').exists() else Path('data')
RAW_PATH = DATA_DIR / 'data.parquet'
TRAIN_PATH = DATA_DIR / 'train.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'
DATASET_PATH = DATA_DIR / 'dataset.parquet'
GOLD_PATH = DATA_DIR / 'gold_labels.parquet'

## Load Data

Prefer `dataset.parquet` when available because it contains weak labels and engineered features. Fall back to `train.parquet` for raw train-set EDA.

In [ ]:
# Load the richest available file for EDA.
path = DATASET_PATH if DATASET_PATH.exists() else TRAIN_PATH
df = pd.read_parquet(path, engine='pyarrow')
raw = pd.read_parquet(RAW_PATH, engine='pyarrow') if RAW_PATH.exists() else None
gold = pd.read_parquet(GOLD_PATH, engine='pyarrow') if GOLD_PATH.exists() else None

print(f'EDA file: {path}')
print('df shape:', df.shape)
print('raw shape:', None if raw is None else raw.shape)
print('gold shape:', None if gold is None else gold.shape)
df.head()

In [ ]:
# Quick schema overview: columns, dtypes, and non-null counts.
schema = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null': df.notna().sum(),
    'null': df.isna().sum(),
    'null_pct': df.isna().mean().mul(100).round(2),
})
schema.sort_values(['null_pct', 'dtype'], ascending=[False, True])

## Similarity Features

These plots should show whether nearest-neighbor similarity features are informative and whether positive/negative neighbor gaps differ by label.

In [ ]:
# Graph should show similarity feature distributions by final label.
sim_cols = [c for c in ['max_similarity', 'mean_similarity', 'top_k_pos_similarity', 'top_k_neg_similarity', 'gap_similarity'] if c in df.columns]
if sim_cols:
    n = len(sim_cols)
    fig, axes = plt.subplots(n, 1, figsize=(9, 3 * n))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, sim_cols):
        if 'label' in df.columns:
            for label, part in df.groupby('label'):
                part[col].dropna().plot(kind='hist', bins=40, alpha=0.45, ax=ax, label=f'label={label}')
            ax.legend()
        else:
            df[col].dropna().plot(kind='hist', bins=40, ax=ax)
        ax.set_title(f'{col} distribution')
        ax.set_xlabel(col)
    plt.tight_layout()
    plt.show()
else:
    print('No similarity feature columns found. Run make train to generate dataset.parquet with similarity features.')

## Topic Features

These plots should show whether BERTTopic found meaningful clusters or mostly assigned outliers (`topic = -1`).

In [ ]:
# Graph should show top topic IDs and the outlier rate.
if 'topic' in df.columns:
    topic_counts = df['topic'].value_counts().head(25).sort_values()
    fig, ax = plt.subplots(figsize=(8, 6))
    topic_counts.plot(kind='barh', ax=ax)
    ax.set_title('Top topic assignments')
    ax.set_xlabel('rows')
    ax.set_ylabel('topic')
    plt.tight_layout()
    plt.show()
    print('Outlier topic share:', (df['topic'] == -1).mean())
else:
    print('No topic column found. Run make train to generate topic features.')

In [ ]:
# Graph should show topic probability confidence distribution.
if 'topic_probability' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    df['topic_probability'].dropna().plot(kind='hist', bins=40, ax=ax)
    ax.set_title('Topic probability distribution')
    ax.set_xlabel('topic probability')
    ax.set_ylabel('rows')
    plt.tight_layout()
    plt.show()